### Phase 3: Modellierung (Der Vergleich)

Hier werden die konkurrierenden Modelle implementiert:

#### 1. Markov-Switching-Modell (MSM)
*   **Library:** `statsmodels.tsa.regime_switching.markov_regression`
*   **Logik:** Ein statistisches Modell, das Wahrscheinlichkeiten für Regimes berechnet.

#### 2. Hidden-Markov-Models (HMM)
*   **Library:** `hmmlearn.hmm`
*   **Logik:** Unsupervised Learning (Clustering), das Zeitabschnitte mit ähnlichen statistischen Verteilungen gruppiert, um verborgene Marktregimes zu identifizieren.

#### 3. LSTM-Netzwerk
*   **Library:** `TensorFlow/Keras` oder `PyTorch`.
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 60 Tage der Features).
    *   Layer: LSTM-Layer -> Dropout -> Dense (Softmax).
    
#### 4. Transformer-Netzwerk
*   **Library:** `PyTorch` (`torch.nn.TransformerEncoder`)
*   **Architektur:**
    *   Input: Zeitreihen-Fenster (z.B. die letzten 60 Tage der Features).
    *   Layer: Linear Projection → Positional Encoding → TransformerEncoder (Multi-Head Self-Attention) → Dense (Sigmoid).
*   **Zweck:** Test von Hypothese H2 — Attention-Mechanismus vs. ökonometrische MSM.

Modelle die ein Feedback (gelabelte Daten) benötigen, um Regime zu erkennen, erhalten diese durch das genauste Modell (im Projektverlauf ermittelt) -> Aktuell: Markov-Switching (Univariat)

---

**Ausführungs-Modus:**
- `cfg.walk_forward.enabled = false` → klassischer 80/20-Split (Cells 4–7).
- `cfg.walk_forward.enabled = true`  → Walk-Forward-Validierung (rollierende Folds), 
  überschreibt die Ergebnisse von Cells 4–7 durch OOS-Vorhersagen.

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
# --- Regime Signal Validation (Helper) ---
# --- Gemeinsame Funktionen aus src/ ---
sys.path.insert(0, "..")
from src.models.common import BEAR_REGIME, BULL_REGIME, validate_regime_signal, create_sequences

# --- Sanity Check ---
# Wird zu Ende jeder nachfolgenden Model-Cell aufgerufen
def validate_regime_signal(data, model_name: str, auto_invert: bool = True) -> None:
    """
    Standardized sanity check for regime signals.
    Expects {model_name}_Prob and {model_name}_Signal in data.
    """
    prob_col = f'{model_name}_Prob'
    signal_col = f'{model_name}_Signal'
    stats_cols = ['Returns', 'VIX', 'Yield_Spread', prob_col]

    # Regime Statistics
    print(f"\n{'='*60}")
    print(f"   {model_name} — Regime-Statistik")
    print(f"{'='*60}")
    available = [c for c in stats_cols if c in data.columns]
    print(data.groupby(signal_col)[available].mean())
    print(f"\nSignal-Verteilung:\n{data[signal_col].value_counts()}")

    # Plausibility Check
    mean_returns = data.groupby(signal_col)['Returns'].mean()
    if mean_returns.get(BEAR_REGIME, 0) > mean_returns.get(BULL_REGIME, 0):
        print(f"\n WARNUNG: {model_name} Bear-Regime ({BEAR_REGIME}) "
              f"hat höhere Returns als Bull ({BULL_REGIME})!")
        if auto_invert:
            print("    → Labels könnten vertauscht sein. Invertiere:")
            data[signal_col] = 1 - data[signal_col]
            data[prob_col] = 1 - data[prob_col]
            print("    → Labels automatisch invertiert.")
    else:
        print(f"\n{model_name} Plausibilitäts-Check bestanden.")

    # Validierung
    assert prob_col in data.columns, f"{prob_col} fehlt!"
    assert signal_col in data.columns, f"{signal_col} fehlt!"
    assert data[signal_col].isin([BULL_REGIME, BEAR_REGIME]).all(), "Signal enthält ungültige Werte!"
    assert data[signal_col].isna().sum() == 0, "NaN im Signal!"
    assert data[prob_col].between(0, 1).all(), "Prob außerhalb [0,1]!"
    print("Alle formalen Prüfungen bestanden.")

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
df = pd.read_parquet(cfg.data_path("feature_engineered"))

In [ ]:
# =============================================================================
# 1. Markov-Switching-Modelle (MS Univariate)
# Typ: Ökonometrie (Regression)
# Beschreibung: Zustandsabhängige Regressionsmodelle mit switching variance.
#               MS Univariate: Nur Returns.
# Config-Key: models.msm
# =============================================================================

import matplotlib.pyplot as plt
import warnings
import os

MODEL_NAME = "MSM"

# Im Walk-Forward-Modus übernimmt run_walk_forward das Training.
if cfg.walk_forward.enabled:
    print(f"{MODEL_NAME}: Skipped — walk_forward.enabled = true.")
else:
    # --- MSM aus src/ ---
    from src.models.msm import train_msm, load_msm, predict_msm
    
    # Warnung ignorieren
    warnings.filterwarnings("ignore")
    
    # --- 0. Hyperparameter aus zentraler Config laden ---
    msm_cfg = cfg.models.msm
    persist = cfg.model_persistence
    model_file = cfg.model_path("msm")
    
    # --- 1. Vorbereitung: Index auf Business Days setzen ---
    df.index = pd.DatetimeIndex(df.index).to_period('B')
    
    if persist.enabled and os.path.exists(model_file):
        # ================================================================
        # MODUS A: Gespeichertes Modell laden (Training überspringen)
        # ================================================================
        print(f"{MODEL_NAME}: Lade persistiertes Modell aus {model_file}")
        ms_uni_results = load_msm(
            model_file=model_file,
            returns=df['Returns'],
            k_regimes=msm_cfg.k_regimes,
            switching_variance=msm_cfg.switching_variance,
        )
    else:
        # ================================================================
        # MODUS B: Normal trainieren + Modell speichern
        # ================================================================
        print(f"{MODEL_NAME}: Starte Training...")
        ms_uni_results = train_msm(
            returns=df['Returns'],
            k_regimes=msm_cfg.k_regimes,
            switching_variance=msm_cfg.switching_variance,
            model_file=model_file,
        )
    
    # --- Wahrscheinlichkeiten und Signal ableiten ---
    df[f'{MODEL_NAME}_Prob'], df[f'{MODEL_NAME}_Signal'] = predict_msm(
        ms_results=ms_uni_results,
        threshold=msm_cfg.threshold,
    )
    
    # --- ABSCHLUSS ---
    # Index wieder zurück in normales Datetime-Format für Plotting
    df.index = df.index.to_timestamp()
    
    print("Markov-Switching-Modell erfolgreich berechnet.")
    
    # --- VISUALISIERUNG ---
    from src.models.plots import plot_msm_regimes
    
    model_color = cfg.color_map.get(MODEL_NAME, 'tab:blue')
    msm_plot_path = cfg.asset_path("markov_model")
    plot_msm_regimes(df, MODEL_NAME, model_color, msm_plot_path)
    
    from IPython.display import Image, display
    display(Image(filename=msm_plot_path))
    
    print(df)
    validate_regime_signal(df, MODEL_NAME)

In [ ]:
# =============================================================================
# 2. Hidden Markov Model (HMM)
# Typ: Unsupervised (Clustering)
# Beschreibung: Identifiziert Regime-Cluster über Gaussian-Emissions in
#               Returns, VIX und Yield_Spread ohne gelabelte Daten.
# Config-Key: models.hmm
# =============================================================================

import matplotlib.pyplot as plt
import os

MODEL_NAME = "HMM"

# Im Walk-Forward-Modus übernimmt run_walk_forward das Training.
if cfg.walk_forward.enabled:
    print(f"{MODEL_NAME}: Skipped — walk_forward.enabled = true.")
else:
    # --- HMM aus src/ ---
    from src.models.hmm import train_hmm, load_hmm, predict_hmm
    
    # --- 0. Hyperparameter aus zentraler Config laden ---
    hmm_cfg = cfg.models.hmm
    persist = cfg.model_persistence
    model_file = cfg.model_path("hmm")
    scaler_file = cfg.model_path("scaler_hmm")
    
    # --- 1. Features vorbereiten ---
    # Returns (Performance), VIX (Angst) und Yield_Spread (Makro)
    hmm_features = hmm_cfg.features
    
    if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
        # ================================================================
        # MODUS A: Gespeichertes Modell laden (Training überspringen)
        # ================================================================
        print(f"{MODEL_NAME}: Lade persistiertes Modell aus {model_file}")
        model_hmm, scaler_hmm, X_hmm_scaled = load_hmm(
            features_df=df[hmm_features],
            model_file=model_file,
            scaler_file=scaler_file,
        )
    else:
        # ================================================================
        # MODUS B: Normal trainieren + Modell speichern
        # ================================================================
        print(f"{MODEL_NAME}: Starte Training...")
        model_hmm, scaler_hmm, X_hmm_scaled = train_hmm(
            features_df=df[hmm_features],
            n_components=hmm_cfg.n_components,
            covariance_type=hmm_cfg.covariance_type,
            n_iter=hmm_cfg.n_iter,
            random_state=hmm_cfg.random_state,
            model_file=model_file,
            scaler_file=scaler_file,
        )
    
    # --- Wahrscheinlichkeiten und Signal ableiten ---
    df[f'{MODEL_NAME}_Prob'], df[f'{MODEL_NAME}_Signal'] = predict_hmm(
        model=model_hmm,
        X_scaled=X_hmm_scaled,
        returns=df['Returns'],
        threshold=hmm_cfg.threshold,
    )
    
    # --- Visualisierung ---
    from src.models.plots import plot_hmm_regimes
    
    model_color = cfg.color_map.get(MODEL_NAME, 'tab:purple')
    hmm_plot_path = cfg.asset_path("hmm_regimes")
    plot_hmm_regimes(df, MODEL_NAME, model_color, hmm_plot_path)
    
    from IPython.display import Image, display
    display(Image(filename=hmm_plot_path))
    
    validate_regime_signal(df, MODEL_NAME)

In [ ]:
# =============================================================================
# 3. LSTM-Netzwerk (Supervised Regime Classification)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: LSTM-Netzwerk mit rollendem Fenster für zeitreihenbasierte
#               Regime-Klassifikation. Labels stammen aus dem MS_Univariate-Modell.
# Config-Key: models.lstm
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import os

MODEL_NAME = "LSTM"

# Im Walk-Forward-Modus übernimmt run_walk_forward das Training.
if cfg.walk_forward.enabled:
    print(f"{MODEL_NAME}: Skipped — walk_forward.enabled = true.")
else:
    # --- LSTM aus src/ ---
    from src.models.lstm import train_lstm, load_lstm_model, predict_lstm
    
    # --- 0. Hyperparameter aus zentraler Config laden ---
    lstm_cfg = cfg.models.lstm
    persist = cfg.model_persistence
    model_file = cfg.model_path("lstm")
    scaler_file = cfg.model_path("scaler_lstm")
    
    # --- 1. Features vorbereiten ---
    # Wir nehmen alle relevanten Informationen für ein "ganzheitliches" Bild
    features = cfg.features.model_features
    print(f"{MODEL_NAME} nutzt folgende Features: {features}")
    
    window_size = lstm_cfg.window_size
    
    if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
        # ================================================================
        # MODUS A: Gespeichertes Modell laden (Training überspringen)
        # ================================================================
        model_lstm, scaler, lstm_probs_raw, split = load_lstm_model(
            df=df,
            features=features,
            labels_col=lstm_cfg.labels,
            window_size=window_size,
            train_test_split=lstm_cfg.train_test_split,
            model_file=model_file,
            scaler_file=scaler_file,
        )
    else:
        # ================================================================
        # MODUS B: Normal trainieren + Modell speichern
        # ================================================================
        print(f"{MODEL_NAME}: Starte Training...")
        model_lstm, scaler, lstm_probs_raw, split = train_lstm(
            df=df,
            features=features,
            labels_col=lstm_cfg.labels,
            window_size=window_size,
            train_test_split=lstm_cfg.train_test_split,
            units_l1=lstm_cfg.units_l1,
            units_l2=lstm_cfg.units_l2,
            return_sequences=lstm_cfg.return_sequences,
            dropout=lstm_cfg.dropout,
            dense=lstm_cfg.dense,
            activation=lstm_cfg.activation,
            optimizer=lstm_cfg.optimizer,
            metrics=lstm_cfg.metrics,
            epochs=lstm_cfg.epochs,
            batch_size=lstm_cfg.batch_size,
            validation_split=lstm_cfg.validation_split,
            verbose=lstm_cfg.verbose,
            model_file=model_file,
            scaler_file=scaler_file,
        )
    
    # --- Wahrscheinlichkeiten und Signal ableiten ---
    probs, signal = predict_lstm(
        probs_raw=lstm_probs_raw,
        threshold=lstm_cfg.threshold,
    )
    
    # --- Test-DataFrame für Backtesting und Visualisierung vorbereiten ---
    # Wir schneiden das df so zu, dass es exakt zu den X_test Daten passt
    test_df = df.iloc[split + window_size:].copy()
    
    # Wahrscheinlichkeiten und Signale speichern
    test_df[f'{MODEL_NAME}_Prob'] = probs
    test_df[f'{MODEL_NAME}_Signal'] = signal
    
    # --- Visualisierung ---
    from src.models.plots import plot_dl_model
    
    model_color = cfg.color_map.get(MODEL_NAME, 'tab:green')
    lstm_plot_path = cfg.asset_path("lstm_model")
    plot_dl_model(test_df, MODEL_NAME, model_color, lstm_plot_path)
    
    from IPython.display import Image, display
    display(Image(filename=lstm_plot_path))
    
    print(test_df)
    validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
# =============================================================================
# 4. Transformer-Netzwerk (Regime Detection via Multi-Head Self-Attention)
# Typ: Supervised (Labels von MS_Univariate)
# Beschreibung: Transformer-Encoder mit Positional Encoding für zeitreihenbasierte
#               Regime-Klassifikation. Testet Hypothese H2 (Attention > Econometric MSM).
# Config-Key: models.transformer
# =============================================================================

import matplotlib.pyplot as plt
import os

MODEL_NAME = "Transformer"

# Im Walk-Forward-Modus übernimmt run_walk_forward das Training.
if cfg.walk_forward.enabled:
    print(f"{MODEL_NAME}: Skipped — walk_forward.enabled = true.")
else:
    # --- Transformer aus src/ ---
    from src.models.transformer import train_transformer, load_transformer_model, predict_transformer
    
    # --- 0. Hyperparameter aus zentraler Config laden ---
    t_cfg = cfg.models.transformer
    persist = cfg.model_persistence
    model_file = cfg.model_path("transformer")
    scaler_file = cfg.model_path("scaler_transformer")
    
    # --- 1. Features vorbereiten ---
    features = cfg.features.model_features
    print(f"Transformer nutzt folgende Features: {features}")
    
    window_size_t = t_cfg.window_size
    
    if persist.enabled and os.path.exists(model_file) and os.path.exists(scaler_file):
        # ================================================================
        # MODUS A: Gespeichertes Modell laden (Training überspringen)
        # ================================================================
        model_transformer, scaler_transformer, transformer_probs_raw, split_t = load_transformer_model(
            df=df,
            features=features,
            labels_col=t_cfg.labels,
            window_size=window_size_t,
            train_test_split=t_cfg.train_test_split,
            d_model=t_cfg.d_model,
            n_heads=t_cfg.n_heads,
            n_layers=t_cfg.n_layers,
            dim_feedforward=t_cfg.dim_feedforward,
            dropout=t_cfg.dropout,
            model_file=model_file,
            scaler_file=scaler_file,
        )
    else:
        # ================================================================
        # MODUS B: Normal trainieren + Modell speichern
        # ================================================================
        print(f"{MODEL_NAME}: Starte Training...")
        model_transformer, scaler_transformer, transformer_probs_raw, split_t = train_transformer(
            df=df,
            features=features,
            labels_col=t_cfg.labels,
            window_size=window_size_t,
            train_test_split=t_cfg.train_test_split,
            d_model=t_cfg.d_model,
            n_heads=t_cfg.n_heads,
            n_layers=t_cfg.n_layers,
            dim_feedforward=t_cfg.dim_feedforward,
            dropout=t_cfg.dropout,
            learning_rate=t_cfg.learning_rate,
            epochs=t_cfg.epochs,
            batch_size=t_cfg.batch_size,
            validation_split=t_cfg.validation_split,
            verbose=t_cfg.verbose,
            model_file=model_file,
            scaler_file=scaler_file,
        )
    
    # --- Wahrscheinlichkeiten und Signal ableiten ---
    probs, signal = predict_transformer(
        probs_raw=transformer_probs_raw,
        threshold=t_cfg.threshold,
    )
    
    # --- Ergebnisse in test_df schreiben ---
    test_df[f'{MODEL_NAME}_Prob'] = probs
    test_df[f'{MODEL_NAME}_Signal'] = signal
    
    # --- Visualisierung ---
    from src.models.plots import plot_dl_model
    
    model_color = cfg.color_map.get(MODEL_NAME, 'darkorange')
    transformer_plot_path = cfg.asset_path("transformer_model")
    plot_dl_model(test_df, MODEL_NAME, model_color, transformer_plot_path)
    
    from IPython.display import Image, display
    display(Image(filename=transformer_plot_path))
    
    validate_regime_signal(test_df, MODEL_NAME)

In [ ]:
# =============================================================================
# 5. Walk-Forward-Validierung (wenn aktiv)
# =============================================================================
# Im Walk-Forward-Modus wird test_df hier neu gebaut: rollierende Re-Trainings,
# OOS-Aggregation, kein Look-Ahead. Die Cells 4–7 oben werden in diesem Fall
# übersprungen.
# =============================================================================

if cfg.walk_forward.enabled:
    import hashlib
    from src.backtest.walk_forward import (
        walk_forward_splits,
        summarize_splits,
        assert_no_leakage,
        run_walk_forward,
        _walk_forward_fingerprint,
        load_walk_forward_cache,
        save_walk_forward_cache,
    )

    print("=" * 60)
    print(" Walk-Forward-Validierung aktiviert")
    print("=" * 60)
    print(f"  Modus: {cfg.walk_forward.mode}")
    print(f"  Train-Fenster: {cfg.walk_forward.train_window_years} Jahre")
    print(f"  Test-Fenster:  {cfg.walk_forward.test_window_months} Monate")
    print(f"  Step:          {cfg.walk_forward.step_months} Monate")
    print()

    # --- 1. Splits generieren ---
    splits = walk_forward_splits(
        index=df.index,
        mode=cfg.walk_forward.mode,
        train_window_years=cfg.walk_forward.train_window_years,
        test_window_months=cfg.walk_forward.test_window_months,
        step_months=cfg.walk_forward.step_months,
        min_train_years=cfg.walk_forward.min_train_years,
    )
    assert_no_leakage(splits)
    print(f"  → {len(splits)} Folds generiert.")

    # --- 2. Übersichtstabelle (Sanity-Check) ---
    splits_summary = summarize_splits(splits)
    display(splits_summary.head())
    display(splits_summary.tail())
    print(f"\n  Train-Größen: min={splits_summary['n_train'].min()}, "
          f"median={int(splits_summary['n_train'].median())}, "
          f"max={splits_summary['n_train'].max()}")
    print(f"  Test-Größen:  min={splits_summary['n_test'].min()}, "
          f"median={int(splits_summary['n_test'].median())}, "
          f"max={splits_summary['n_test'].max()}")

    # --- 3. Cache prüfen oder Walk-Forward ausführen ---
    cache_path = cfg.data_path("walk_forward_cache")
    idx_hash = hashlib.sha256(
        df.index.astype(str).str.cat().encode()
    ).hexdigest()[:16]
    fingerprint = _walk_forward_fingerprint(cfg, df.shape, idx_hash)
    print(f"\n  Walk-Forward-Fingerprint: {fingerprint}")

    use_cache = getattr(cfg.walk_forward, "cache_enabled", False)
    cached_df = None
    if use_cache:
        cached_df = load_walk_forward_cache(cache_path, fingerprint)

    if cached_df is not None:
        test_df = cached_df
        print(f"  → Cache-Hit! Überspringe Training ({len(test_df)} OOS-Zeilen).")
    else:
        # --- Walk-Forward ausführen (dauert!) ---
        test_df = run_walk_forward(
            df=df,
            splits=splits,
            cfg=cfg,
            models_to_run=["MSM", "HMM", "LSTM", "Transformer"],
        )

        # --- 4. OOS-Bereich isolieren (Train-only-Zeilen verwerfen) ---
        signal_cols = [c for c in test_df.columns if c.endswith("_Signal")]
        test_df = test_df.dropna(subset=signal_cols, how="all").copy()

        # Cache speichern
        if use_cache:
            save_walk_forward_cache(test_df, fingerprint, cache_path)

    # Returns + Cash_Returns müssen im test_df enthalten bleiben (für Backtest).
    # Da wir df kopiert haben, sind sie automatisch dabei.
    print(f"\n  test_df OOS-Bereich: {test_df.index.min().date()} → "
          f"{test_df.index.max().date()} ({len(test_df)} Tage)")

    # --- 5. Validierung pro Modell ---
    signal_cols = [c for c in test_df.columns if c.endswith("_Signal")]
    for model_name in ["MSM", "HMM", "LSTM", "Transformer"]:
        sig_col = f"{model_name}_Signal"
        if sig_col not in signal_cols:
            continue
        # Pro Modell separat dropna, da LSTM/Transformer wegen window_size
        # später beginnen als MSM/HMM
        sub = test_df.dropna(subset=[sig_col]).copy()
        if len(sub) == 0:
            print(f"\n  WARNUNG: {model_name} hat keine OOS-Vorhersagen!")
            continue
        validate_regime_signal(sub, model_name)

else:
    print("Walk-Forward deaktiviert (cfg.walk_forward.enabled = false). "
          "Verwende Single-Split-Ergebnisse aus Cells 4–7.")

In [ ]:
# Im Walk-Forward-Modus: für den Vergleichsplot auf den gemeinsamen OOS-Bereich
# beschneiden, damit alle Modelle dieselben Zeitachsen-Ticks haben.
if cfg.walk_forward.enabled:
    signal_cols = [c for c in test_df.columns if c.endswith("_Signal")]
    test_df_plot = test_df.dropna(subset=signal_cols, how="any")
else:
    test_df_plot = test_df

from src.models.plots import plot_regime_comparison

comparison_path = cfg.asset_path("regime_comparison")
plot_regime_comparison(test_df, cfg.color_map, comparison_path)

from IPython.display import Image, display
display(Image(filename=comparison_path))

In [ ]:
output_path = cfg.data_path("test_data")

# Speichern als Parquet
test_df.to_parquet(output_path)

print(f"Dataframe erfolgreich unter {output_path} gespeichert.")